# ETL and transformation

What most scheduled jobs actually do: deduplicate, compare against yesterday, reshape, parse,
apply business rules, clean up text.

Several of these work on the defects planted in notebook 00. **T1** removes the duplicate rows,
**T8** collapses `'UK'`, `'uk'` and `' Uk '` into one value, and **T2** works out what changed
between two snapshots.

**T3** pivots with `CASE` rather than either engine's `PIVOT` syntax, which keeps the SQL
identical on both sides. **T6** could not be kept identical because the JSON functions are named
differently.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "05_etl_transform", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")


In [ ]:
SQL_T1 = """
SELECT count(*) AS kept FROM (
  SELECT sale_id, row_number() OVER (PARTITION BY sale_id ORDER BY sale_ts) AS rn
  FROM sales
) t WHERE rn = 1
"""

_, out, _ = bench.run(Case("T1", "Deduplicate (keep first per key)", "ETL", sql=SQL_T1.strip()))
display(out.head())


In [ ]:
SQL_T2 = """
SELECT
  sum(CASE WHEN v2.sale_id IS NULL THEN 1 ELSE 0 END)                       AS deleted,
  sum(CASE WHEN v2.sale_id IS NOT NULL AND
       COALESCE(v2.amount,-1) <> COALESCE(s.amount,-1) THEN 1 ELSE 0 END)   AS changed,
  sum(CASE WHEN v2.sale_id IS NOT NULL AND
       COALESCE(v2.amount,-1) =  COALESCE(s.amount,-1) THEN 1 ELSE 0 END)   AS unchanged
FROM sales s LEFT JOIN sales_v2 v2 ON v2.sale_id = s.sale_id
"""

_, out, _ = bench.run(Case("T2", "Snapshot diff (what changed)", "ETL", sql=SQL_T2.strip()))
display(out.head())


In [ ]:
SQL_T3 = """
SELECT region,
  round(sum(CASE WHEN month(sale_date) = 1 THEN amount ELSE 0 END),2) AS jan,
  round(sum(CASE WHEN month(sale_date) = 2 THEN amount ELSE 0 END),2) AS feb,
  round(sum(CASE WHEN month(sale_date) = 3 THEN amount ELSE 0 END),2) AS mar,
  round(sum(CASE WHEN month(sale_date) = 4 THEN amount ELSE 0 END),2) AS apr
FROM sales GROUP BY 1 ORDER BY 1
"""

_, out, _ = bench.run(Case("T3", "Pivot (months into columns)", "ETL", sql=SQL_T3.strip()))
display(out.head())


In [ ]:
SQL_T4 = """
SELECT metric, round(sum(value),2) AS total FROM (
  SELECT 'amount' AS metric, amount AS value FROM sales
  UNION ALL SELECT 'unit_price', unit_price FROM sales
  UNION ALL SELECT 'discount_pct', discount_pct FROM sales
) t GROUP BY 1 ORDER BY 1
"""

_, out, _ = bench.run(Case("T4", "Unpivot (columns into rows)", "ETL", sql=SQL_T4.strip()))
display(out.head())


In [ ]:
SQL_T5 = """
SELECT build, count(*) AS n FROM (
  SELECT regexp_extract(user_agent, 'build ([0-9]+)', 1) AS build FROM sales
) t GROUP BY 1 ORDER BY n DESC LIMIT 10
"""

_, out, _ = bench.run(Case("T5", "Regex parsing from text", "ETL", sql=SQL_T5.strip()))
display(out.head())


In [ ]:
SQL_T6_DUCK = """
SELECT json_extract_string(attributes_json, '$.experiment') AS experiment, count(*) AS n, round(avg(CAST(json_extract_string(attributes_json,'$.score') AS DOUBLE)),3) AS avg_score FROM sales GROUP BY 1 ORDER BY 1
"""

SQL_T6_SPARK = """
SELECT get_json_object(attributes_json, '$.experiment') AS experiment, count(*) AS n, round(avg(CAST(get_json_object(attributes_json,'$.score') AS DOUBLE)),3) AS avg_score FROM sales GROUP BY 1 ORDER BY 1
"""

_, out, _ = bench.run(Case("T6", "Parse JSON column", "ETL", duck_sql=SQL_T6_DUCK.strip(), spark_sql=SQL_T6_SPARK.strip()))
display(out.head())


In [ ]:
SQL_T7 = """
SELECT band, tier, count(*) AS n, round(sum(net),2) AS net_total FROM (
  SELECT
    CASE WHEN amount IS NULL THEN 'unknown' WHEN amount < 0 THEN 'invalid'
         WHEN amount < 50 THEN 'low' WHEN amount < 200 THEN 'medium'
         WHEN amount < 500 THEN 'high' ELSE 'vip' END AS band,
    CASE WHEN used_promo = 1 AND discount_pct > 0.2 THEN 'deep_promo'
         WHEN used_promo = 1 THEN 'promo'
         WHEN channel IN ('web','app') THEN 'digital' ELSE 'other' END AS tier,
    COALESCE(amount,0) * (1 - discount_pct) * (1 + tax_pct) AS net
  FROM sales
) t GROUP BY 1,2 ORDER BY 1,2
"""

_, out, _ = bench.run(Case("T7", "Business rules (CASE heavy)", "ETL", sql=SQL_T7.strip()))
display(out.head())


In [ ]:
SQL_T8 = """
SELECT upper(trim(country)) AS country_clean, count(*) AS customers
FROM customers GROUP BY 1 ORDER BY 1
"""

_, out, _ = bench.run(Case("T8", "Clean and standardise text", "ETL", sql=SQL_T8.strip()))
display(out.head())


In [ ]:
SQL_T9 = """
SELECT CAST(date_trunc('month', sale_date) AS DATE) AS month,
       count(*) AS orders, round(sum(amount),2) AS revenue
FROM sales GROUP BY 1 ORDER BY 1
"""

_, out, _ = bench.run(Case("T9", "Date bucketing", "ETL", sql=SQL_T9.strip()))
display(out.head())


## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()
